# Family Quant AI - Data Explorer

Interactive notebook for exploring market data, features, earnings, and macro indicators.

**Run `python update_market_data.py` then `python scripts/feature_engine.py` first.**

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

conn = duckdb.connect('../data/market_data.duckdb', read_only=True)

tables = conn.execute('SHOW TABLES').fetchdf()
print('Tables in database:')
for t in tables['name']:
    count = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {count:,} rows')

## 1. Stock Price History

In [ ]:
# Compare all tickers normalized to 100
all_prices = conn.execute("""
    SELECT symbol, timestamp::DATE as date, close
    FROM daily_bars ORDER BY symbol, timestamp
""").fetchdf()

plt.figure(figsize=(14, 6))
for sym in ['TSLA', 'AAPL', 'NVDA', 'SPY']:
    data = all_prices[all_prices['symbol'] == sym].copy()
    data['normalized'] = data['close'] / data['close'].iloc[0] * 100
    plt.plot(data['date'], data['normalized'], label=sym)

plt.title('All Tickers: Normalized Performance (Start = 100)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Feature Dashboard

In [ ]:
# TSLA feature summary
tsla_feat = conn.execute("""
    SELECT * FROM daily_features
    WHERE symbol = 'TSLA'
    ORDER BY date
""").fetchdf()

print(f'TSLA features: {len(tsla_feat)} days, {tsla_feat.shape[1]} columns')
print(f'Date range: {tsla_feat["date"].min()} to {tsla_feat["date"].max()}')
print()
print('Latest values:')
print(tsla_feat.tail(1).T)

In [ ]:
# Key technical features over time
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

axes[0,0].plot(tsla_feat['date'], tsla_feat['return_20d'] * 100)
axes[0,0].set_title('20-Day Return (%)')
axes[0,0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(tsla_feat['date'], tsla_feat['volatility_20d'] * 100)
axes[0,1].set_title('20-Day Volatility (annualized %)')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(tsla_feat['date'], tsla_feat['rsi_14'])
axes[1,0].set_title('RSI (14-day)')
axes[1,0].axhline(y=70, color='red', linestyle='--', alpha=0.5)
axes[1,0].axhline(y=30, color='green', linestyle='--', alpha=0.5)
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(tsla_feat['date'], tsla_feat['bollinger_position'])
axes[1,1].set_title('Bollinger Band Position (-1 to +1)')
axes[1,1].axhline(y=1, color='red', linestyle='--', alpha=0.5)
axes[1,1].axhline(y=-1, color='green', linestyle='--', alpha=0.5)
axes[1,1].grid(True, alpha=0.3)

axes[2,0].plot(tsla_feat['date'], tsla_feat['relative_volume'])
axes[2,0].set_title('Relative Volume (vs 20d avg)')
axes[2,0].axhline(y=1, color='gray', linestyle='--', alpha=0.5)
axes[2,0].grid(True, alpha=0.3)

axes[2,1].plot(tsla_feat['date'], tsla_feat['relative_strength_vs_spy'])
axes[2,1].set_title('Relative Strength vs SPY (20d)')
axes[2,1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[2,1].grid(True, alpha=0.3)

plt.suptitle('TSLA Technical Features', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Regime Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].plot(tsla_feat['date'], tsla_feat['vix'])
axes[0,0].set_title('VIX')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(tsla_feat['date'], tsla_feat['fed_funds'])
axes[0,1].set_title('Fed Funds Rate (%)')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(tsla_feat['date'], tsla_feat['treasury_10y'])
axes[1,0].set_title('10-Year Treasury Yield (%)')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(tsla_feat['date'], tsla_feat['days_since_earnings'])
axes[1,1].set_title('Days Since Last Earnings')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('TSLA Regime + Fundamental Features', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Earnings & Price Moves

In [ ]:
tsla_earnings = conn.execute("""
    SELECT e.reported_date, e.surprise_pct,
        (SELECT close FROM daily_bars WHERE symbol='TSLA' AND timestamp::DATE <= e.reported_date ORDER BY timestamp DESC LIMIT 1) as close_before,
        (SELECT close FROM daily_bars WHERE symbol='TSLA' AND timestamp::DATE > e.reported_date ORDER BY timestamp LIMIT 1) as close_after
    FROM earnings e
    WHERE e.symbol = 'TSLA' AND e.reported_date >= '2020-01-01'
    ORDER BY e.reported_date
""").fetchdf()

tsla_earnings['price_move_pct'] = (tsla_earnings['close_after'] / tsla_earnings['close_before'] - 1) * 100

plt.figure(figsize=(12, 5))
colors = ['green' if x > 0 else 'red' for x in tsla_earnings['price_move_pct']]
plt.bar(range(len(tsla_earnings)), tsla_earnings['price_move_pct'], color=colors, alpha=0.7)
plt.title('TSLA: Price Move After Each Earnings Report')
plt.ylabel('Next-Day Price Move (%)')
plt.xlabel('Earnings Report #')
plt.axhline(y=0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Average absolute move: {tsla_earnings["price_move_pct"].abs().mean():.1f}%')
print(f'Positive moves: {(tsla_earnings["price_move_pct"] > 0).sum()} / {len(tsla_earnings)}')

## 5. Annualized Returns

In [ ]:
returns = conn.execute("""
    SELECT symbol,
        MIN(timestamp)::DATE as start_date, MAX(timestamp)::DATE as end_date,
        FIRST(close ORDER BY timestamp) as first_close,
        LAST(close ORDER BY timestamp) as last_close
    FROM daily_bars GROUP BY symbol
""").fetchdf()

returns['total_return'] = returns['last_close'] / returns['first_close'] - 1
returns['calendar_days'] = (pd.to_datetime(returns['end_date']) - pd.to_datetime(returns['start_date'])).dt.days
returns['annualized'] = (1 + returns['total_return']) ** (365 / returns['calendar_days']) - 1

print('Annualized Returns (365 calendar days, comparable to CD rates):')
print('=' * 55)
for _, row in returns.iterrows():
    print(f"  {row['symbol']}: {row['annualized']*100:+.1f}% per year  (total: {row['total_return']*100:+.1f}%)")

In [ ]:
conn.close()
print('Done.')